# 📊 Exploración de Fuentes y Análisis de Casuística de Fallos (Capa Bronze)

## 1. Introducción y Objetivos
Este Notebook está destinado al **Análisis Exploratorio de Datos (EDA)** del parque eólico Kelmarsh. El objetivo principal es auditar la calidad de los datos en la capa **Bronze** antes de consolidar las transformaciones de producción hacia la capa **Silver**.

Específicamente nos centraremos en:
1. **Inspección Estructural:** Documentar el esquema y consistencia de los logs de estado (`Status`).
2. **Análisis de Casuística de Fallos:** Clasificar y limpiar los más de 7,000 eventos para determinar cuáles representan paradas críticas reales y cuáles son alertas operativas o paradas programadas.
3. **Optimización del Target (PdM):** Evaluar la ventana de degradación temporal óptima para el etiquetado del modelo predictivo de IA.

---

## 2. Inicialización del Entorno Spark
Configuramos una sesión local optimizada explotando la paralelización del hardware disponible.

In [6]:
import os
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Inicializar Spark optimizado para el entorno local
spark = SparkSession.builder \
    .appName("Kelmarsh-EDA-Notebook") \
    .master("local[6]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# Definir rutas relativas desde la carpeta de notebooks
BASE_DIR = os.path.dirname(os.getcwd())
BRONZE_DIR = os.path.join(BASE_DIR, "data", "bronze")

# Cargar logs de Status de la Turbina 1
status_files = glob.glob(f"{BRONZE_DIR}/**/Status_Kelmarsh_1_*.csv", recursive=True)
status_df = spark.read \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .option("comment", "#") \
    .csv(status_files)

# Mostrar metadatos esenciales
status_df.printSchema()
status_df.show(5, truncate=False)

root
 |-- Timestamp start: timestamp (nullable = true)
 |-- Timestamp end: string (nullable = true)
 |-- Duration: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Code: integer (nullable = true)
 |-- Message: string (nullable = true)
 |-- Comment: string (nullable = true)
 |-- Service contract category: string (nullable = true)
 |-- IEC category: string (nullable = true)

+-------------------+-------------------+---------+-------------+----+--------------------------------+-------+-----------------------------------+----------------------------------+
|Timestamp start    |Timestamp end      |Duration |Status       |Code|Message                         |Comment|Service contract category          |IEC category                      |
+-------------------+-------------------+---------+-------------+----+--------------------------------+-------+-----------------------------------+----------------------------------+
|2016-12-17 16:38:07|2017-01-09 12:51:27|548:13:20|Warni

---

## 3. Introducción al Análisis de Códigos y Mensajes del SCADA

Antes de agrupar y filtrar a ciegas, necesitamos entender qué está registrando realmente el sistema de control de la turbina. El objetivo de esta sección es extraer la lista completa de códigos únicos presentes en la capa Bronze.

Analizaremos la combinación de:
* El mensaje de error (`Message`).
* La categoría contractual (`Service contract category`).
* La norma internacional de energía eólica (`IEC category`).

Esto nos permitirá concluir con datos reales qué eventos son averías críticas y cuáles son simples paradas operacionales o caídas de comunicación.

In [11]:
# Contar y ordenar todos los tipos de eventos para ver la casuística real
status_df.groupBy("Code", "Message", "Status", "Service contract category", "IEC category") \
         .count() \
         .orderBy(F.desc("count")) \
         .show(40, truncate=False)

+------+---------------------------------+-------------+-----------------------------------+----------------------------------+-----+
|Code  |Message                          |Status       |Service contract category          |IEC category                      |count|
+------+---------------------------------+-------------+-----------------------------------+----------------------------------+-----+
|0     |System OK                        |Informational|System OK (32)                     |Full Performance                  |1916 |
|10    |Wind < start wind                |Informational|External stop (low wind speed)  (5)|Out of Environmental Specification|1554 |
|100130|Automatic start-up               |Informational|NULL                               |Full Performance                  |861  |
|100180|Run-up                           |Informational|NULL                               |Technical Standby                 |852  |
|100200|Mains run-up                     |Informational|NULL  

Tras analizar el listado de frecuencias y categorías IEC del SCADA, clasificamos los eventos en tres categorías operacionales para definir la lógica de ventana del `target`:

| Tipo de Evento | Códigos de Muestra | Categoría IEC / Contrato | Acción para el Target Predictivo |
| :--- | :--- | :--- | :--- |
| **Operación Normal / Clima** | `0`, `10`, `100130`, `100180`,`...` | *Full Performance / Out of Env. Spec* | **Ignorar (Ventana = 0h).** La máquina está sana o parada por falta de viento. |
| **Rutinas y Mantenimiento** | `710`, `20`, `210`,`...`  | *Technical Standby / Scheduled Maint.* | **Ignorar (Ventana = 0h).** Son paradas manuales en el sitio o tests automáticos. |
| **Fallo Crítico / Degradación** | `5720`, `2125`,`...`  | *Warnings (27)* | **Etiquetar Ventana de Alerta (Ej. 24h antes).** Son anomalías en componentes (ej. sistema de frenado) que avisan con desgaste previo. |

---

## 4. Aislamiento de Todos los Eventos Críticos (Warnings y Stops)

Para evitar pérdidas catastróficas, no podemos limitar el análisis a los códigos más frecuentes. Un único evento de fallo crítico (aunque tenga un `count = 1`) puede suponer la rotura de un componente mayor y costes económicos muy elevados.

En esta sección, filtraremos el dataset completo para extraer de forma hermética **todos** los códigos catalogados como `Warning` o `Stop`, excluyendo explícitamente las rutinas operacionales conocidas (como los tests de batería o las paradas manuales de mantenimiento programado).

In [17]:
# Lista de códigos operacionales o manuales que ya sabemos que NO son fallos
codigos_operacionales = [710, 20, 210, 8000]

# Filtrar para dejar solo Warnings y Stops reales que requieran atención predictiva
alertas_criticas_df = status_df.filter(
    (F.col("Status").isin("Warning", "Stop")) & 
    (~F.col("Code").isin(codigos_operacionales))
)

# Agrupar para ver el catálogo completo de amenazas reales presentes en el histórico
alertas_criticas_df.groupBy("Code", "Message", "Status", "Service contract category") \
    .count() \
    .orderBy(F.asc("count")) \
    .show(100, truncate=False)

+----+--------------------------------------+-------+--------------------------------------+-----+
|Code|Message                               |Status |Service contract category             |count|
+----+--------------------------------------+-------+--------------------------------------+-----+
|7325|Time sync. failed (SNTP error)        |Warning|Warnings (27)                         |1    |
|6525|4-20mA anemometer 2                   |Warning|Warnings (27)                         |1    |
|801 |Rotor sensor A defective              |Stop   |Sensor error (21)                     |1    |
|6635|4-20 mA vane 2                        |Warning|Sensor error (21)                     |1    |
|682 |Limit switch error 95° axis 2         |Warning|Warnings (27)                         |1    |
|3585|Maximum grid frequency                |Stop   |External stop (grid) (4)              |1    |
|3501|Grid disconnection for self-protection|Stop   |External stop (grid) (4)              |1    |
|1825|Over

La inspección de eventos singulares (`count = 1`) demuestra que los fallos de baja frecuencia (como partículas en el aceite de la multiplicadora o fallos en sensores de rotor) poseen un impacto económico crítico y no pueden ignorarse. Sin embargo, el catálogo puro de averías internas sigue mostrando "ruido" operativo externo y maniobras de control (ej. caídas de red o paradas remotas).

Para evitar un mantenimiento manual e ineficiente de listas de exclusión código a código, se requiere diseñar una metodología de filtrado escalable combinando patrones léxicos (búsqueda de palabras clave como "manual") y agrupaciones por categorías estructurales (`Service contract category` e `IEC category`).

---

## 5. Micro-Segmentación y Criba de Eventos Singulares (Count = 1)

El análisis de eventos de baja frecuencia confirma que no podemos ignorar las alertas por tener un bajo conteo. Sin embargo, para diseñar el `target` de degradación mecánica definitivo, debemos separar los eventos residuales en tres naturalezas:

### 1. ⚠️ Averías Mecánicas/Físicas Reales (A monitorizar por la IA)
Son fallos internos que muestran un desgaste del activo. El modelo predictivo **SÍ** debe anticiparse a ellos:
* **Multiplicadora (Gear):** `1825` (Sobrecarga de filtro bypass) y `1922` (Alarma de partículas metálicas en aceite).
* **Sensores Críticos:** `801` (Fallo sensor del rotor) y `1620` (Velocidad de la multiplicadora implausible).
* **Sistemas de Orientación (Yaw/Axis):** `682` (Fallo de carrera en el eje).

### 2. ⚡ Eventos Eléctricos Externos (Fuerza Mayor)
La turbina se detiene por anomalías en la red eléctrica exterior. La máquina está sana. **Ventana = 0h**:
* `3585`, `3501`, `3590` (*External stop grid* / Sobrevoltaje / Desconexión por autoprotección).
* `9210` (*Externally stopped*).

### 3. 🛠️ Maniobras Humanas o de Seguridad Manual
Paradas provocadas por el personal de mantenimiento en campo. **Ventana = 0h**:
* `111` (*Emergency stop nacelle* - Seta de emergencia pulsada en la góndola).
* `117` (*Emergency stop base box* - Seta de emergencia en la base de la torre).

In [31]:
# 1. Filtro Léxico: Identificar palabras clave operacionales o humanas en el mensaje
palabras_excluir = ["manual", "test", "ok", "unwind", "flushing", "stopped", "sync"]
filtro_lexico = " | ".join(palabras_excluir)

# 2. Filtro Estructural: Excluir categorías completas del contrato o estándar IEC que no son averías mecánicas
categorias_excluir = [
    "External stop (grid) (4)",
    "External stop (low wind speed)  (5)",
    "Remote stop (30)",
    "System OK (32)",
    "Operating states  (28)"
]

# 3. Aplicar el filtrado inteligente combinado
catálogo_fallos_reales_df = status_df.filter(
    # Queremos registrar fallos reales que detienen la máquina o avisan de degradación
    (F.col("Status").isin("Warning", "Stop")) & 
    
    # Capa 1: Eliminar cualquier mensaje que contenga palabras operativas (Ignorando mayúsculas/minúsculas)
    (~F.col("Message").rlike(f"(?i)({filtro_lexico})")) & 
    
    # Capa 2: Eliminar categorías contractuales completas de paradas externas o normales
    (~F.col("Service contract category").isin(categorias_excluir)) &
    
    # Capa 3: Eliminar por seguridad los códigos operacionales que ya aislamos al principio
    (~F.col("Code").isin([111, 117, 707, 710, 20, 210, 8000]))
)

# Guardar y auditar la lista definitiva de los códigos que SÍ importan
print("🎯 CATÁLOGO AUTOMATIZADO DE AVERÍAS MECÁNICAS E INTERNAS:")
catalogo_final = catálogo_fallos_reales_df.groupBy("Code", "Message", "Status", "Service contract category", "IEC category") \
    .count() \
    .orderBy(F.desc("count"))

catalogo_final.show(100, truncate=False)

🎯 CATÁLOGO AUTOMATIZADO DE AVERÍAS MECÁNICAS E INTERNAS:
+----+-----------------------------------+-------+-----------------------------------+-------------------+-----+
|Code|Message                            |Status |Service contract category          |IEC category       |count|
+----+-----------------------------------+-------+-----------------------------------+-------------------+-----+
|5720|Brake accumulator defect           |Warning|Warnings (27)                      |NULL               |71   |
|2125|Timeout brake closed               |Warning|Warnings (27)                      |NULL               |37   |
|8400|Comm. failure FPM                  |Warning|Warnings (27)                      |Full Performance   |27   |
|805 |High rotor speed nacelle           |Stop   |Overspeed (14)                     |Forced outage      |17   |
|2550|Overload generator fan 1           |Warning|Warnings (27)                      |Forced outage      |11   |
|5000|Breakdown obstacle light         

In [32]:
# ==============================================================================
# CAPA DE LIMPIEZA Y CONFIGURACIÓN DE FILTROS INTELIGENTES (CONSOLIDADO)
# ==============================================================================

# 1. Exclusiones por Código ID (Rutinas conocidas, paradas manuales y de seguridad)
codigos_operacionales_y_setas = [
    710,   # Battery test
    20,    # Manual stop - on site
    210,   # Manual brake
    8000,  # Park master stop (Parada remota de red)
    707,   # Stop battery test
    111,   # Emergency stop nacelle
    117    # Emergency stop base box
]

# 2. Exclusiones por Categorías Contractuales/IEC Completas (Clima y Red Externa)
categorias_no_mecanicas = [
    "External stop (grid) (4)",
    "External stop (low wind speed)  (5)",
    "Remote stop (30)",
    "System OK (32)",
    "Operating states  (28)"
]

# 3. Exclusiones Léxicas por Expresiones Regulares (Luz de avión, caídas de IT, etc.)
palabras_ruido_auxiliar = ["manual", "test", "ok", "unwind", "flushing", "stopped", "sync", "obstacle", "comm.", "pmu", "light"]
filtro_lexico_regex = f"(?i)({'|'.join(palabras_ruido_auxiliar)})"

# ==============================================================================
# APLICACIÓN DEL FILTRADO INTEGRAL EN UN SOLO BLOQUE
# ==============================================================================
catalogo_fallos_reales_df = status_df.filter(
    # Filtrar por criticidad base (Solo nos interesan anomalías y paradas forzosas)
    (F.col("Status").isin("Warning", "Stop")) &
    
    # Capa 1: Filtrar IDs específicos de mantenimiento humano y maniobras de control
    (~F.col("Code").isin(codigos_operacionales_y_setas)) &
    
    # Capa 2: Purga de categorías externas (viento bajo, caídas de la red general)
    (~F.col("Service contract category").isin(categorias_no_mecanicas)) &
    
    # Capa 3: Hachazo léxico automático a las bombillas fundidas y enlaces de telecomunicaciones
    (~F.col("Message").rlike(filtro_lexico_regex))
)

# ==============================================================================
# AUDITORÍA DE VOLUMETRÍA SIN CORTES (...)
# ==============================================================================
# Agrupar y ordenar el catálogo final
catalogo_final = catalogo_fallos_reales_df.groupBy(
    "Code", "Message", "Status", "Service contract category", "IEC category"
).count().orderBy(F.desc("count"))

# Calcular el número exacto de fallos mecánicos reales residuales
num_averias_unicas = catalogo_final.count()

print(f"📊 EL CATÁLOGO FILTRADO TIENE EXACTAMENTE: {num_averias_unicas} CÓDIGOS DE AVERÍA ÚNICOS.\n")
print("🎯 LISTA MAESTRA DEFINITIVA DE AVERÍAS MECÁNICAS E INTERNAS (VISTA COMPLETA):")

# Forzamos a show a mostrar el total exacto para que jamás se vuelva a cortar con (...)
catalogo_final.show(n=num_averias_unicas + 5, truncate=False)



📊 EL CATÁLOGO FILTRADO TIENE EXACTAMENTE: 38 CÓDIGOS DE AVERÍA ÚNICOS.

🎯 LISTA MAESTRA DEFINITIVA DE AVERÍAS MECÁNICAS E INTERNAS (VISTA COMPLETA):
+----+-----------------------------------+-------+-----------------------------------+-------------------+-----+
|Code|Message                            |Status |Service contract category          |IEC category       |count|
+----+-----------------------------------+-------+-----------------------------------+-------------------+-----+
|5720|Brake accumulator defect           |Warning|Warnings (27)                      |NULL               |71   |
|2125|Timeout brake closed               |Warning|Warnings (27)                      |NULL               |37   |
|805 |High rotor speed nacelle           |Stop   |Overspeed (14)                     |Forced outage      |17   |
|2550|Overload generator fan 1           |Warning|Warnings (27)                      |Forced outage      |11   |
|2655|Overload generator fan 3           |Warning|Warnings (

El volumen masivo de datos operacionales y de infraestructura se ha reducido drásticamente a **exactamente 38 códigos de avería únicos**. Pasar de un histórico de miles de alertas ruidosas a una lista de este tamaño representa un éxito crítico en la fase de ingeniería de características (*feature engineering*): el dataset ya es **completamente manejable**.

Disponer de un catálogo acotado a 38 filas nos otorga la ventaja estratégica de poder realizar una **auditoría visual exhaustiva, línea por línea**, asegurando que ningún "intruso" operacional se escape a los filtros lógicos masivos. Dado que las interfaces de visualización de los cuadernos suelen truncar las tablas medianas, este volumen optimizado es el tamaño perfecto para exportar un reporte limpio en formato texto (`.txt`) que sirva como control de calidad definitivo antes de avanzar hacia el etiquetado del target de Machine Learning.

In [33]:
# Definir la ruta del archivo de texto en la raíz de tu carpeta de notebooks
ruta_auditoria = "catalogo_definitivo_auditoria.txt"

# Extraer las filas locales para la escritura
filas_catalogo = catalogo_final.collect()

with open(ruta_auditoria, "w", encoding="utf-8") as f:
    f.write(f"📋 CATALOGO TOTAL DE AVERIAS PARA REVISION (38 EVENTOS)\n")
    f.write("=" * 100 + "\n")
    f.write(f"{'CÓDIGO':<8} | {'MENSAJE':<40} | {'STATUS':<8} | {'CATEGORÍA CONTRACTUAL'}\n")
    f.write("-" * 100 + "\n")
    
    for fila in filas_catalogo:
        f.write(f"{str(fila['Code']):<8} | {str(fila['Message']):<40} | {str(fila['Status']):<8} | {str(fila['Service contract category'])}\n")

print(f"✅ ¡Archivo generado con éxito! Abre '{ruta_auditoria}' desde el explorador de VS Code para ver las 38 líneas completas.")


✅ ¡Archivo generado con éxito! Abre 'catalogo_definitivo_auditoria.txt' desde el explorador de VS Code para ver las 38 líneas completas.


Gracias a este análisis meticuloso línea por línea, se ha detectado un "intruso" operativo camuflado que los filtros automáticos categóricos no habían logrado purgar por completo:

* **`Code 9000 - P output externally reduced` (Warning):** Este evento no representa una degradación física, mecánica ni eléctrica intrínseca del aerogenerador. Se trata de una consigna de reducción de potencia impuesta desde el exterior (por el operador del sistema eléctrico o limitaciones de red).

**Acción definitiva:** Tras identificar y aislar el código `9000`, se ha incorporado de forma fulminante a la lista maestra de exclusiones operacionales. Con esta última corrección de ingeniería, el catálogo se reduce de forma estricta y limpia a **exactamente 37 códigos de avería puros**. El dataset queda 100% validado, libre de sesgos externos y optimizado para la fase de cálculo de duraciones y etiquetado del target de Machine Learning.

In [ ]:
# ==============================================================================
# CAPA DE LIMPIEZA Y CONFIGURACIÓN DE FILTROS INTELIGENTES (CONSOLIDADO FINAL)
# ==============================================================================

# 1. Exclusiones por Código ID (Rutinas, setas de emergencia y consignas externas)
codigos_operacionales_y_setas = [
    710,   # Battery test
    20,    # Manual stop - on site
    210,   # Manual brake
    8000,  # Park master stop (Parada remota de red)
    707,   # Stop battery test
    111,   # Emergency stop nacelle
    117,   # Emergency stop base box
    9000   # P output externally reduced (Consigna externa de reducción de potencia) 
]

# 2. Exclusiones por Categorías Contractuales/IEC Completas (Clima y Red Externa)
categorias_no_mecanicas = [
    "External stop (grid) (4)",
    "External stop (low wind speed)  (5)",
    "Remote stop (30)",
    "System OK (32)",
    "Operating states  (28)"
]

# 3. Exclusiones Léxicas por Expresiones Regulares (Luz de avión, caídas de IT, etc.)
palabras_ruido_auxiliar = ["manual", "test", "ok", "unwind", "flushing", "stopped", "sync", "obstacle", "comm.", "pmu", "light"]
filtro_lexico_regex = f"(?i)({'|'.join(palabras_ruido_auxiliar)})"

# ==============================================================================
# APLICACIÓN DEL FILTRADO INTEGRAL DEFINITIVO
# ==============================================================================
catalogo_fallos_reales_df = status_df.filter(
    (F.col("Status").isin("Warning", "Stop")) &
    (~F.col("Code").isin(codigos_operacionales_y_setas)) &
    (~F.col("Service contract category").isin(categorias_no_mecanicas)) &
    (~F.col("Message").rlike(filtro_lexico_regex))
)

# Cacheamos este DataFrame porque va a ser la base sagrada de nuestro Target
catalogo_fallos_reales_df.cache()

# Re-generar agrupación para verificar el número final
catalogo_final = catalogo_fallos_reales_df.groupBy(
    "Code", "Message", "Status", "Service contract category", "IEC category"
).count().orderBy(F.desc("count"))

print(f"📊 AHORA EL CATÁLOGO TIENE EXACTAMENTE: {catalogo_final.count()} CÓDIGOS DE AVERÍA ELECTROMECÁNICA INTRÍNSECOS.")

📊 AHORA EL CATÁLOGO TIENE EXACTAMENTE: 37 CÓDIGOS DE AVERÍA MECÁNICA PUROS.


26/05/29 12:03:51 WARN CacheManager: Asked to cache already cached data.


## 9. Definición Conceptual del Target: Fallos Electromecánicos Intrínsecos

Es imperativo corregir la nomenclatura del catálogo definitivo. Los 37 códigos resultantes no corresponden en exclusiva a fallos mecánicos de desgaste, sino a un conjunto de **Averías Electromecánicas Intrínsecas del Activo**.

En la ingeniería de un aerogenerador, las fronteras entre los subsistemas están acopladas:
1. **Origen Eléctrico / Electrónico:** Componentes como el convertidor de frecuencia (`3110`), las sobrecargas en los ventiladores (`2550`) o los fallos en los lazos de carga de baterías (`716`) poseen una naturaleza puramente eléctrica.
2. **Impacto Mecánico/Térmico:** Si estos sistemas fallan, desencadenan de forma inmediata pérdidas de refrigeración, desequilibrios de par o fallos de actuación que destruyen componentes físicos (multiplicadora, rodamientos, ejes).

**Conclusión para el modelo predictivo:** Al limpiar el ruido externo (red eléctrica exterior y clima), la IA no se limita a predecir "roturas de piezas", sino que monitoriza la **salud electromecánica interna e integral** de la máquina.